# Fine-tune Qwen3.5-4B for the Decillion dsh agentic system

Run the cells in order (Runtime → Change runtime type → a GPU; A100/L4 recommended, T4 works with shorter context).

1. GPU check
2. Mount Google Drive (outputs go to `MyDrive/decillion-qwen35-4b`)
3. Install libraries
4. Configuration (the only cell you normally edit)
5. Clone the dataset from GitHub and build the training rows
6. Choose the training mix
7. Load Qwen3.5-4B with LoRA
8. Render rows with the model's chat template
9. Trainer with loss on the assistant's turns only
10. Train (checkpoints, including the LoRA adapter, are saved to Drive and resumed after a disconnect)
11. Evaluate: validation loss and perplexity on the agent's turns (saved to Drive)
12. Save a merged 16-bit model to Drive (for serving) and free the GPU
13. Serve the merged model with vLLM: an OpenAI-compatible API on port 8000
14. Expose that API to the internet with a Cloudflare quick tunnel
15. Test the public API end to end and print the provider config for Decillion

Cells 13–15 need the GPU runtime and run after training. The quick tunnel URL is temporary: it lives
only while this runtime and `cloudflared` stay up, so rerun cells 13–14 (and update the provider in
Decillion) after a disconnect.

In [1]:
# 1. GPU check
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → GPU"
GPU = torch.cuda.get_device_name(0)
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(GPU, f"{GPU_MEM_GB:.0f} GB")

Sat Nov 21 17:20:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Google Drive: checkpoints, logs and the final model live here
from google.colab import drive
drive.mount('/content/drive')
import os
OUT_DIR = '/content/drive/MyDrive/decillion-qwen35-4b'
os.makedirs(OUT_DIR, exist_ok=True)
print('outputs →', OUT_DIR)

Mounted at /content/drive
outputs → /content/drive/MyDrive/decillion-qwen35-4b


In [3]:
# 3. Install (Unsloth brings matching torch / transformers / trl / peft versions)
!pip install -q -U unsloth unsloth_zoo
!pip install -q -U jsonschema
# If loading the model later fails on the Qwen3.5 architecture, run this and restart the runtime:
# !pip install -q -U transformers

In [4]:
# 4. Configuration (the only cell you normally edit)
MODEL_ID = 'Qwen/Qwen3.5-4B'
REPO = 'https://github.com/keyhmoham/Aseman.git'
BRANCH = 'main'
DATA_DIR = '/content/Aseman/datasets/decillion-dsh-agentic'

BIG_GPU = GPU_MEM_GB >= 35
MAX_SEQ_LEN = 16384 if BIG_GPU else 8192
LOAD_IN_4BIT = not BIG_GPU

# LoRA
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 32, 0.0

# Training
EPOCHS = 1
LEARNING_RATE = 1e-4
BATCH_SIZE = 2 if BIG_GPU else 1
GRAD_ACCUM = 8 if BIG_GPU else 16
WARMUP_RATIO = 0.03
SAVE_STEPS = 200

# Data mix: multi-turn sessions are ~94% of the data; single-turn families are repeated to balance it
SINGLE_TURN_REPEAT = 4
MAX_MULTI_TURN = 30000   # cap on session.* rows per epoch (None = all ~64k)
TOOLS_MODE = 'relevant'  # 'relevant' (default) or 'full' (all ~130 tool schemas per row; needs long context)
SEED = 3407

print(dict(MODEL_ID=MODEL_ID, MAX_SEQ_LEN=MAX_SEQ_LEN, LOAD_IN_4BIT=LOAD_IN_4BIT, BATCH=BATCH_SIZE, ACCUM=GRAD_ACCUM))

{'MODEL_ID': 'Qwen/Qwen3.5-4B', 'MAX_SEQ_LEN': 8192, 'LOAD_IN_4BIT': True, 'BATCH': 1, 'ACCUM': 16}


In [5]:
# 5. Get the dataset from GitHub and build the training rows (about 3 minutes)
import os
if not os.path.exists(DATA_DIR):
    !git clone -q --depth 1 -b {BRANCH} {REPO} /content/Aseman
else:
    !git -C /content/Aseman pull -q
!python3 {DATA_DIR}/scripts/build.py --tools {TOOLS_MODE} --out /content/dist
!ls -la /content/dist
!cat /content/dist/stats.json | head -40

{"train": 66918, "validation": 1421}
total 977384
drwxr-xr-x 2 root root      4096 Nov 21 17:27 .
drwxr-xr-x 1 root root      4096 Nov 21 17:22 ..
-rw-r--r-- 1 root root      2670 Nov 21 17:27 stats.json
-rw-r--r-- 1 root root 980101154 Nov 21 17:27 train.jsonl.gz
-rw-r--r-- 1 root root  20717530 Nov 21 17:27 validation.jsonl.gz
{
 "train": {
  "total": 66918,
  "by_category": {
   "api.confined": 1,
   "api.policy_denied": 2,
   "api.search_call": 35,
   "apps.file_intent_denied": 1,
   "apps.function": 58,
   "apps.image_generate": 6,
   "apps.mcp_add": 5,
   "apps.plain_words": 12,
   "apps.routine_daily": 32,
   "apps.untrusted_output": 2,
   "appui.build_publish": 15,
   "appui.offer_first": 1,
   "dev.bugfix": 513,
   "dev.edit_replace_all": 17,
   "dev.read_long_file": 4,
   "dev.script_deliver": 29,
   "memory.disabled": 1,
   "memory.recall": 96,
   "memory.recall_miss": 2,
   "memory.remember": 118,
   "memory.rewrite_notes": 1,
   "memory.session_search": 2,
   "memory.timel

In [6]:
# 6. Choose the training mix (rows stay raw JSON lines to save memory)
import gzip, json, random, re
from collections import Counter
random.seed(SEED)
CAT = re.compile(r"\"category\": \"([^\"]+)\"")
single, multi = [], []
for line in gzip.open('/content/dist/train.jsonl.gz', 'rt', encoding='utf-8'):
    cat = CAT.search(line[:300]).group(1)
    (multi if cat.startswith('session.') else single).append(line)
random.shuffle(multi)
if MAX_MULTI_TURN:
    multi = multi[:MAX_MULTI_TURN]
train_lines = single * SINGLE_TURN_REPEAT + multi
random.shuffle(train_lines)
val_lines = gzip.open('/content/dist/validation.jsonl.gz', 'rt', encoding='utf-8').read().splitlines()
print(f'single-turn {len(single)} x{SINGLE_TURN_REPEAT}, multi-turn {len(multi)} -> train {len(train_lines)}, validation {len(val_lines)}')

single-turn 2280 x4, multi-turn 30000 -> train 39120, validation 1421


In [7]:
# 7. Load Qwen3.5-4B and attach LoRA adapters
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)
model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1121 17:28:58.334000 5665 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1121 17:28:58.391000 5665 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.9: Fast Qwen3_5 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
trainable params: 42,467,328 || all params: 4,581,732,864 || trainable%: 0.9269


In [8]:
# 8. Render each row with the model's own chat template (tools included) and drop rows that do not fit
from datasets import Dataset
tok = getattr(tokenizer, "tokenizer", tokenizer)

def render(line):
    row = json.loads(line)
    return tokenizer.apply_chat_template(row["messages"], tools=row["tools"], tokenize=False)

def rows(lines):
    for line in lines:
        text = render(line)
        if len(tok(text, add_special_tokens=False).input_ids) <= MAX_SEQ_LEN:
            yield {"text": text}

train_ds = Dataset.from_generator(rows, gen_kwargs={"lines": train_lines})
val_ds = Dataset.from_generator(rows, gen_kwargs={"lines": val_lines})
print('train', len(train_ds), 'of', len(train_lines), '| validation', len(val_ds), 'of', len(val_lines))
print(train_ds[0]["text"][-1500:])

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

train 2553 of 39120 | validation 102 of 1421
se><|im_end|>
<|im_start|>assistant
Tap the link under this message and press **Start** in Telegram — it works once, for ten minutes.<|im_end|>
<|im_start|>user
Research the five best-reviewed coworking spaces in Lisbon with prices, and make me a comparison doc.<|im_end|>
<|im_start|>assistant
<tool_call>
<function=hand_to_workspace>
<parameter=request>
Research the five best-reviewed coworking spaces in Lisbon with prices, and make me a comparison doc. Deliver the result as a file and a short summary.
</parameter>
</function>
</tool_call><|im_end|>
<|im_start|>user
<tool_response>
Started in Home (space 0000000083a18a14456fc7a863222b7b). The result will be written in this chat when the team finishes: say so; do not wait for it.
</tool_response><|im_end|>
<|im_start|>assistant
It's under way with your workspace team; the result will show up here when they finish.<|im_end|>
<|im_start|>user
<from_your_memory>
- The person's timezone is Europe

In [9]:
# 9. Trainer: loss only on the assistant's turns (replies and tool calls), not on system, user or tool results
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
from unsloth import is_bfloat16_supported

args = SFTConfig(
    output_dir=f'{OUT_DIR}/checkpoints',
    dataset_text_field='text',
    packing=False,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    optim='adamw_8bit',
    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=SAVE_STEPS,
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    report_to='none',
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds.select(range(min(300, len(val_ds)))),
    args=args,
)
# Qwen chat format: tool results arrive inside user turns (<tool_response>), so masking every
# non-assistant turn leaves exactly the agent's own output in the loss.
trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)
print('steps per epoch ≈', len(train_ds) // (BATCH_SIZE * GRAD_ACCUM))

Unsloth: not enough free memory for dataset tokenization workers (~1GB each); tokenizing in-process.
Unsloth: transformers renamed `warmup_ratio` to `warmup_steps`. Forwarding your value to `warmup_steps` - update your code when convenient. If you also passed `warmup_steps` as 0.1, that is its default here and cannot be distinguished from leaving it unset, so `warmup_ratio` was used; drop `warmup_ratio` to keep it.


`causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"]:   0%|          | 0/2553 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"]:   0%|          | 0/102 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/2553 [00:00<?, ? examples/s]

Map:   0%|          | 0/102 [00:00<?, ? examples/s]

steps per epoch ≈ 159


In [ ]:
# 10. Train (resumes from the last checkpoint on Drive if the runtime was disconnected)
import glob
last = sorted(glob.glob(f'{OUT_DIR}/checkpoints/checkpoint-*'), key=lambda p: int(p.rsplit('-', 1)[1]))
stats = trainer.train(resume_from_checkpoint=last[-1] if last else None)
print(stats)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,553 | Num Epochs = 1 | Total steps = 160
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 42,467,328 of 4,581,732,864 (0.93% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss


In [ ]:
# 11. Evaluate on the held-out validation split: loss and perplexity on the agent's own turns
import math, json as _json
metrics = trainer.evaluate(eval_dataset=val_ds)
metrics['eval_perplexity'] = math.exp(metrics['eval_loss'])
print(_json.dumps(metrics, indent=1))
_json.dump(metrics, open(f'{OUT_DIR}/eval_metrics.json', 'w'), indent=1)

In [ ]:
# 12. Save the fine-tuned model as one merged 16-bit model (needed for serving), then free the GPU
import gc, torch
MERGED_DIR = f'{OUT_DIR}/merged'
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')
print('merged model →', MERGED_DIR)
del trainer, model
gc.collect(); torch.cuda.empty_cache()
# After a runtime restart you can skip training: run cells 1, 2, 4, then set
# MERGED_DIR = f'{OUT_DIR}/merged' and continue from cell 13.

In [ ]:
# 13. Serve the merged model with vLLM: an OpenAI-compatible API on port 8000
# After a runtime restart, run cells 1, 2 and 4 first; this cell finds the merged model on Drive.
import json, os, shutil, subprocess, sys, time, urllib.request
from importlib.metadata import PackageNotFoundError, version
import torch

OUT_DIR = globals().get('OUT_DIR', '/content/drive/MyDrive/decillion-qwen35-4b')
MODEL_ID = globals().get('MODEL_ID', 'Qwen/Qwen3.5-4B')
MERGED_DIR = globals().get('MERGED_DIR', f'{OUT_DIR}/merged')
SERVED_NAME = 'decillion-dsh-qwen35-4b'
PORT = 8000
MAX_LEN = max(int(globals().get('MAX_SEQ_LEN', 16384)), 16384)  # the context window to register in Decillion
VLLM_LOG = '/content/vllm.log'

assert os.path.exists(f'{MERGED_DIR}/config.json'), f'no merged model at {MERGED_DIR}: run cell 12 first'

# The merged folder can miss the processor/chat-template files of the multimodal base model.
from huggingface_hub import snapshot_download
base = snapshot_download(MODEL_ID, allow_patterns=[
    'preprocessor_config.json', 'video_preprocessor_config.json', 'processor_config.json',
    'chat_template.jinja', 'generation_config.json'])
for name in os.listdir(base):
    if os.path.isfile(f'{base}/{name}') and not os.path.exists(f'{MERGED_DIR}/{name}'):
        shutil.copy2(f'{base}/{name}', f'{MERGED_DIR}/{name}')

# Serve from local disk: mmap over the Drive FUSE mount is slow and unreliable.
LOCAL_DIR = '/content/merged'
stamp = f'{LOCAL_DIR}/.source-mtime'
src_mtime = os.path.getmtime(f'{MERGED_DIR}/config.json')
stale = (not os.path.exists(f'{LOCAL_DIR}/config.json') or not os.path.exists(stamp)
         or float(open(stamp).read()) < src_mtime)
if stale:
    print('copying the merged model to local disk (a few minutes)…')
    shutil.rmtree(LOCAL_DIR, ignore_errors=True)
    shutil.copytree(MERGED_DIR, LOCAL_DIR)
    open(stamp, 'w').write(str(src_mtime))

try:
    have_vllm = tuple(int(x) for x in version('vllm').split('.')[:2]) >= (0, 34)
except (PackageNotFoundError, ValueError):
    have_vllm = False
if not have_vllm:
    print('installing vLLM (a few minutes; it brings its own torch)…')
    !pip install -q -U 'vllm>=0.34.0'

# The tunnel publishes this port to the internet, so the API always needs a key.
API_KEY = globals().get('API_KEY') or 'sk-decillion-' + os.urandom(16).hex()
# T4 has no bf16; every other Colab GPU does.
DTYPE = 'bfloat16' if torch.cuda.is_bf16_supported() else 'float16'

subprocess.run(['pkill', '-f', 'vllm.entrypoints.openai.api_server'], capture_output=True)
time.sleep(3)
cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', LOCAL_DIR,
    '--served-model-name', SERVED_NAME,
    '--host', '127.0.0.1', '--port', str(PORT),
    '--api-key', API_KEY,
    '--dtype', DTYPE,
    '--max-model-len', str(MAX_LEN),
    '--gpu-memory-utilization', '0.9' if DTYPE == 'float16' else '0.85',
    '--max-num-seqs', '8',
    # dsh's agentic calls: Qwen's XML <tool_call> blocks and its  thinking block
    '--enable-auto-tool-choice', '--tool-call-parser', 'qwen3_xml', '--reasoning-parser', 'qwen3',
    # this fine-tune is text-only: do not reserve memory for images or video
    '--limit-mm-per-prompt', '{"image": 0, "video": 0}',
]
if DTYPE == 'float16':
    cmd += ['--enforce-eager']   # a 16 GB T4: skip the CUDA graphs to keep 16k context in memory
print('starting vLLM…')
proc = subprocess.Popen(cmd, stdout=open(VLLM_LOG, 'w'), stderr=subprocess.STDOUT)

def served_ids():
    req = urllib.request.Request(f'http://127.0.0.1:{PORT}/v1/models',
                                 headers={'Authorization': f'Bearer {API_KEY}'})
    return [m['id'] for m in json.load(urllib.request.urlopen(req, timeout=5))['data']]

deadline, ids = time.time() + 900, None
while time.time() < deadline and proc.poll() is None:
    try:
        ids = served_ids()
        break
    except Exception:
        time.sleep(5)
if ids != [SERVED_NAME]:
    print(open(VLLM_LOG).read()[-4000:])
    raise RuntimeError('vLLM did not come up: see its log above')

SERVE = {'base_url_local': f'http://127.0.0.1:{PORT}/v1', 'port': PORT, 'model': SERVED_NAME,
         'api_key': API_KEY, 'max_model_len': MAX_LEN, 'merged_dir': MERGED_DIR}
json.dump(SERVE, open(f'{OUT_DIR}/serve.json', 'w'), indent=1)
print('serving', SERVED_NAME, '→', SERVE['base_url_local'])
print('API key:', API_KEY, '| saved to', f'{OUT_DIR}/serve.json')

In [ ]:
# 14. Expose the vLLM API to the internet with a Cloudflare quick tunnel (no account needed)
import json, os, re, subprocess, time, urllib.request

OUT_DIR = globals().get('OUT_DIR', '/content/drive/MyDrive/decillion-qwen35-4b')
SERVE = json.load(open(f'{OUT_DIR}/serve.json'))
assert 'base_url_local' in SERVE, 'run cell 13 first'
PORT, API_KEY = SERVE['port'], SERVE['api_key']
CF, CF_LOG = '/usr/local/bin/cloudflared', '/content/cloudflared.log'

if not os.path.exists(CF):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {CF}
    !chmod +x {CF}

# Make sure the model is still up before publishing it.
req = urllib.request.Request(f'http://127.0.0.1:{PORT}/v1/models',
                             headers={'Authorization': f'Bearer {API_KEY}'})
urllib.request.urlopen(req, timeout=10).read()

subprocess.run(['pkill', '-f', 'cloudflared tunnel'], capture_output=True)
time.sleep(2)
if os.path.exists(CF_LOG):
    os.remove(CF_LOG)
# QUIC (cloudflared's default) is often blocked on Colab; HTTP/2 goes over TCP 443.
subprocess.Popen([CF, 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate',
                  '--protocol', 'http2', '--logfile', CF_LOG],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

url, deadline = None, time.time() + 90
while time.time() < deadline and not url:
    time.sleep(2)
    if os.path.exists(CF_LOG):
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open(CF_LOG).read())
        url = m.group(0) if m else None
if not url:
    print(open(CF_LOG).read()[-2000:] if os.path.exists(CF_LOG) else '')
    raise RuntimeError('cloudflared printed no URL')
print('public URL:', url)

def public_ids():
    req = urllib.request.Request(f'{url}/v1/models', headers={'Authorization': f'Bearer {API_KEY}'})
    return [m['id'] for m in json.load(urllib.request.urlopen(req, timeout=10))['data']]

deadline = time.time() + 120
while True:
    try:
        ids = public_ids()
        break
    except Exception as exc:
        if time.time() > deadline:
            raise RuntimeError(f'the tunnel is up but the API is not reachable yet: {exc}')
        time.sleep(3)

SERVE['base_url_public'] = url + '/v1'
json.dump(SERVE, open(f'{OUT_DIR}/serve.json', 'w'), indent=1)
print('reachable from the internet →', SERVE['base_url_public'], '| model', ids[0])
print('This URL lives only while this runtime and cloudflared stay up; rerun this cell and update')
print('the provider in Decillion if it changes. A named Cloudflare tunnel (dashboard → token) gives')
print('a stable hostname instead: cloudflared tunnel run --token <TOKEN>.')

In [ ]:
# 15. Test the public API end to end, the way the Decillion LLM gateway uses it
import gzip, json, os, time
import requests
from jsonschema import validate
from openai import OpenAI

OUT_DIR = globals().get('OUT_DIR', '/content/drive/MyDrive/decillion-qwen35-4b')
DATA_DIR = globals().get('DATA_DIR', '/content/Aseman/datasets/decillion-dsh-agentic')
SERVE = json.load(open(f'{OUT_DIR}/serve.json'))
assert 'base_url_public' in SERVE, 'run cell 14 first: the public URL is missing from serve.json'
BASE, KEY, MODEL = SERVE['base_url_public'].rstrip('/'), SERVE['api_key'], SERVE['model']
client = OpenAI(base_url=BASE, api_key=KEY, timeout=300, max_retries=1)
HEADERS = {'Authorization': f'Bearer {KEY}'}

results = []
def check(name, ok, detail=''):
    results.append((name, bool(ok)))
    print(('PASS' if ok else 'FAIL'), '|', name, ('— ' + detail if detail else ''))

# 1. The OpenAI surface the gateway expects, with and without the key.
r = requests.get(f'{BASE}/models', timeout=30)
check('refuses requests without the key', r.status_code == 401, f'HTTP {r.status_code}')
r = requests.get(f'{BASE}/models', headers=HEADERS, timeout=30)
ids = [m.get('id') for m in r.json().get('data', [])] if r.status_code == 200 else []
check('/v1/models lists the fine-tuned model', ids == [MODEL], str(ids))

# 2. A plain completion.
resp = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': 'Reply with exactly: DECILLION-OK'}],
    temperature=0, max_tokens=64, extra_body={'chat_template_kwargs': {'enable_thinking': False}})
text = (resp.choices[0].message.content or '').strip()
check('a chat completion comes back', 'DECILLION-OK' in text, repr(text[:80]))

# 3. The gateway's own probe shape: streaming, with usage, timed.
t0 = time.time()
first = usage = None
streamed = reasoned = ''
body = {'model': MODEL, 'stream': True, 'stream_options': {'include_usage': True},
        'messages': [{'role': 'user', 'content': 'In about 60 words, explain why the sky is blue.'}],
        'max_tokens': 300}
with requests.post(f'{BASE}/chat/completions', json=body, stream=True, timeout=300,
                   headers={**HEADERS, 'Accept': 'text/event-stream'}) as r:
    r.raise_for_status()
    for line in r.iter_lines(decode_unicode=True):
        if not line or not line.startswith('data:'):
            continue
        payload = line[5:].strip()
        if payload == '[DONE]':
            break
        chunk = json.loads(payload)
        if chunk.get('usage'):
            usage = chunk['usage']
        for choice in chunk.get('choices', []):
            delta = choice.get('delta') or {}
            if first is None and (delta.get('content') or delta.get('reasoning_content') or delta.get('tool_calls')):
                first = time.time() - t0
            streamed += delta.get('content') or ''
            reasoned += delta.get('reasoning_content') or ''
check('streams tokens and reports usage for metering',
      first is not None and (streamed.strip() or reasoned.strip()) and usage
      and usage.get('prompt_tokens', 0) > 0 and usage.get('completion_tokens', 0) > 0,
      f'first token {first and round(first, 2)}s, {usage and usage.get("completion_tokens")} completion tokens')
print('   sample:', repr((streamed.strip() or reasoned.strip())[:160]))

# 4. Held-out validation rows: does the served model behave like the dsh agent?
path = '/content/dist/validation.jsonl.gz'
if not os.path.exists(path):
    path = f'{DATA_DIR}/dist/validation.jsonl.gz'
rows = [json.loads(line) for line in gzip.open(path, 'rt', encoding='utf-8')]

def first_expected(row):
    for i, message in enumerate(row['messages']):
        if message['role'] == 'assistant':
            calls = message.get('tool_calls') or []
            return i, (calls[0]['function']['name'] if calls else None)
    return None, None

wanted = ['dev.bugfix', 'people.ask_user', 'memory.remember', 'team.delegate_parallel']
picked = []
for category in wanted:
    for row in rows:
        ai, expected = first_expected(row)
        if row['category'] == category and expected:
            picked.append((row, ai, expected))
            break
check('held-out validation rows are available', len(picked) == len(wanted), f'{len(picked)}/{len(wanted)}')

matched = 0
for row, ai, expected in picked:
    tools = row['tools']
    try:
        resp = client.chat.completions.create(model=MODEL, messages=row['messages'][:ai], tools=tools,
                                              tool_choice='auto', temperature=0, max_tokens=1024)
    except Exception as exc:
        check(f'{row["category"]}: calls {expected}', False, str(exc)[:140])
        continue
    message = resp.choices[0].message
    call = (message.tool_calls or [None])[0]
    got = call.function.name if call else None
    args_ok = False
    if call:
        schema = next((t.get('function', t).get('parameters') for t in tools
                       if t.get('function', t).get('name') == got), None)
        try:
            validate(json.loads(call.function.arguments), schema)
            args_ok = True
        except Exception:
            args_ok = False
    matched += got == expected
    check(f'{row["category"]}: calls {expected}', got == expected and args_ok,
          f'got {got}, arguments valid={args_ok}')
check('behaves like the dsh agent (3 of 4 held-out tool choices)', matched >= 3, f'{matched}/{len(picked)}')

# 5. Tool calls also have to stream as deltas (what dsh consumes).
if picked:
    row, ai, expected = picked[0]
    body = {'model': MODEL, 'messages': row['messages'][:ai], 'tools': row['tools'], 'tool_choice': 'auto',
            'temperature': 0, 'max_tokens': 1024, 'stream': True,
            'stream_options': {'include_usage': True}}
    name = arguments = ''
    with requests.post(f'{BASE}/chat/completions', json=body, stream=True, timeout=300,
                       headers={**HEADERS, 'Accept': 'text/event-stream'}) as r:
        r.raise_for_status()
        for line in r.iter_lines(decode_unicode=True):
            if not line or not line.startswith('data:'):
                continue
            payload = line[5:].strip()
            if payload == '[DONE]':
                break
            for choice in json.loads(payload).get('choices', []):
                for call in (choice.get('delta') or {}).get('tool_calls') or []:
                    fn = call.get('function') or {}
                    name += fn.get('name') or ''
                    arguments += fn.get('arguments') or ''
    check('tool calls stream as OpenAI deltas', name == expected and arguments.lstrip().startswith('{'),
          f'name={name!r} arguments={arguments[:80]!r}')

# 6. The fine-tune, not base Qwen: the dsh persona is in the weights.
system = next((m['content'] for m in rows[0]['messages'] if m['role'] == 'system'), '')
resp = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'system', 'content': system},
                           {'role': 'user', 'content': 'In one short sentence, who are you?'}],
    temperature=0, max_tokens=512)
message = resp.choices[0].message
reasoning = getattr(message, 'reasoning_content', None) or (message.model_extra or {}).get('reasoning_content', '')
answer = ((message.content or '') + ' ' + (reasoning or '')).strip()
check('answers as decillion, not as base Qwen', 'decillion' in answer.lower(), repr(answer[:140]))

# 7. The provider to paste into Decillion (Admin → AI → Providers → Add → Custom).
gateway = {'name': 'Decillion dsh Qwen3.5-4B (Colab vLLM)', 'kind': 'custom', 'base_url': BASE,
           'api_key': KEY, 'enabled': True,
           'models': [{'id': MODEL, 'name': 'Decillion dsh Qwen3.5-4B',
                       'context_window': SERVE['max_model_len'], 'capabilities': ['chat'],
                       'input_price': 0, 'output_price': 0}]}
json.dump(gateway, open(f'{OUT_DIR}/decillion_gateway_provider.json', 'w'), indent=1)
print('\nprovider for Decillion (also saved to decillion_gateway_provider.json):')
print(json.dumps(gateway, indent=1))

failed = [name for name, ok in results if not ok]
print(f'\n{len(results) - len(failed)}/{len(results)} checks passed')
if failed:
    raise AssertionError('failed: ' + '; '.join(failed))
print('The fine-tuned model is served correctly. Keep this runtime (and cloudflared) alive while')
print('the provider is in use; after a disconnect rerun cells 13–14 and update the base URL.')